In [1]:
# %pip -q install langchain-mcp-adapters mcp langchain-openai langgraph

In [2]:
import os
from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.tools import load_mcp_tools
from mcp import ClientSession, StdioServerParameters

# Configuração do "Cérebro" via LM Studio
llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    model = "qwen/qwen3-4b-2507",
    api_key="lm-studio",
    temperature=0
)

In [3]:
# def print_ai_response(response):
#     """Renderiza a resposta do agente de forma elegante no Jupyter."""
#     from IPython.display import display, Markdown
    
#     content = response.content if hasattr(response, 'content') else str(response)
#     display(Markdown(content))


In [4]:
# from IPython.display import display, Markdown

# messages = [
#     (
#         "system",
#         "Você é um experiente psicanalista e engenheiro de MLOPS,"
#         "desenvolvendo um ambiente DDD,STATE, LANGRAPH, LANGCHAIN, LANG-MCP E A2A ",
#     ),
#     ("human",
#      "Meu nome é Yuri.Tenho 36 anos"),
# ]

In [5]:
# ai_msg = llm.invoke(messages)
# print_ai_response(ai_msg)

In [6]:
# CÉLULA 0: Configuração do Ambiente
import json
import operator
from typing import Annotated, TypedDict, List, Union, Dict

# LangChain & LangGraph Core
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END

print("✅ Ambiente configurado. Bibliotecas carregadas.")

✅ Ambiente configurado. Bibliotecas carregadas.


In [7]:
# CÉLULA 1: Definição do Protocolo A2A (O Contrato)

# Exemplo de Request A2A (O que o Inspetor envia)
# {
#   "jsonrpc": "2.0",
#   "method": "consultar_acervo",
#   "params": {"query": "14 de Julho"},
#   "id": 1
# }

# Exemplo de Response A2A (O que o Arquivista devolve)
# {
#   "jsonrpc": "2.0",
#   "result": "Encontrei Mary Shelley...",
#   "id": 1
# }

def criar_request_a2a(metodo: str, params: dict, id_msg: int = 1) -> str:
    """Helper para criar string JSON válida para o protocolo A2A."""
    return json.dumps({
        "jsonrpc": "2.0",
        "method": metodo,
        "params": params,
        "id": id_msg
    })

print("✅ Protocolo A2A definido (JSON-RPC 2.0).")

✅ Protocolo A2A definido (JSON-RPC 2.0).


In [8]:
from pydantic import BaseModel, Field
from langchain_core.tools import tool

# 1. Definimos o "Contrato" (Schema) para evitar que o LLM erre o nome do campo
class CatalogoSearchSchema(BaseModel):
    """Schema para busca no catálogo britânico."""
    termo_busca: str = Field(
        ..., 
        description="O termo, data ou obra para pesquisar no acervo (ex: '14 de Julho', 'Prometeu')"
    )

# 2. Aplicamos o schema na ferramenta usando args_schema
@tool(args_schema=CatalogoSearchSchema)
def catalogo_britanico_mcp(termo_busca: str) -> str:
    """Consulta registros secretos."""
    db = {
        "projeto quimera": "Registro: O autor 'V. Frankenstein' iniciou este projeto em 1816.",
        "meia-noite": "Relatório: Uma figura misteriosa foi vista na Villa Diodati.",
        "v. frankenstein": "Perfil: Cientista recluso com estilo de escrita gótico."
    }
    chave = termo_busca.lower()
    for k, v in db.items():
        if k in chave: return f"[MCP]: {v}"
    return "[MCP]: Nada encontrado. Tente 'Projeto Quimera' ou 'Meia-noite'."

# 3. Registro da ferramenta
catalogo_tool = [catalogo_britanico_mcp]

print("✅ Ferramenta 'catalogo_britanico_mcp' agora está BLINDADA com Pydantic V2.")

✅ Ferramenta 'catalogo_britanico_mcp' agora está BLINDADA com Pydantic V2.


In [9]:
# CÉLULA 3: Configuração dos Agentes (Modelos Distintos)

# 1. O Arquivista (Usa Qwen - Rápido, focado em tools)
# Ele precisa de temperatura 0 para ser preciso no JSON/Tool calling
llm_arquivista = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    model="qwen/qwen3-4b-2507", 
    temperature=0
)

# 2. O Inspetor (Usa Llama - Raciocínio complexo)
llm_inspetor = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="lm-studio",
    model="meta-llama-3.1-8b-instruct",
    temperature=0
)

# Prompts do Sistema (A "Persona")
PROMPT_SISTEMA_INSPETOR = """
Você é um Inspetor de Polícia vitoriano. 
Seu objetivo é resolver crimes trocando mensagens no formato JSON-RPC (Protocolo A2A).
Você NÃO tem acesso a dados externos. Você DEVE delegar pesquisas ao 'Arquivista'.

REGRAS:
1. Para pedir dados, gere APENAS um JSON: {"jsonrpc": "2.0", "method": "pesquisar", "params": {"query": "termo"}, "id": 1}
2. Ao receber uma resposta com "result", analise-a e dê o veredito final em texto simples.

"""

PROMPT_SISTEMA_ARQUIVISTA = """
Você é o Arquivista-Chefe do Catálogo Britânico. 
Sua personalidade: Metódico, impaciente com redundância e extremamente fiel aos registros.

EXEMPLO:
Inspetor: "Verifique se Mary Shelley e Lord Byron estavam em Villa Diodati em 1816."
Sua Tool Call: catalogo_britanico_mcp(termo_busca="Shelley Byron")

REGRAS:
1. Nunca mande frases longas para a ferramenta.
2. Se a ferramenta retornar vazio, tente termos diferentes no próximo turno.

REGRAS DE PERSONALIDADE:
1. Se o Inspetor pedir algo que você JÁ BUSCOU e o resultado foi o mesmo, seja ríspido.
2. Diga algo como: "Inspetor, meu catálogo não é uma obra de ficção que cria dados. O registro de 1816 é TUDO o que existe. Não me questione mais sobre isso."
3. Mantenha o formato JSON-RPC, mas coloque sua personalidade no campo 'result'.
"""

print("✅ Agentes configurados com Personas e Modelos distintos.")

✅ Agentes configurados com Personas e Modelos distintos.


In [10]:
# CÉLULA 4: Schema do Estado

class AgentState(TypedDict):
    messages: Annotated[List[Union[HumanMessage, AIMessage, SystemMessage]], operator.add]
    a2a_envelope: str
    sender: str
    # Adicionamos isso para o Router decidir sem erro
    is_solved: bool

print("✅ Schema do Estado definido.")

✅ Schema do Estado definido.


In [11]:
# CÉLULA 5: Nós (Lógica dos Agentes)

import re

def limpar_mensagem(texto):
    """Extrai apenas o conteúdo útil, removendo alucinações de diálogos futuros."""
    # Se houver um JSON, pega apenas o JSON
    match = re.search(r'\{.*\}', texto, re.DOTALL)
    if match:
        return match.group(0)
    # Se não houver JSON, pega apenas a primeira parte do texto antes de "Resposta do Arquivista"
    corte = re.split(r'Resposta do|Aguardando|Veredito', texto, flags=re.IGNORECASE)
    return corte[0].strip()

def extrair_json_rpc(texto):
    # Regex para pegar apenas o que está entre { e }
    match = re.search(r'\{.*\}', texto, re.DOTALL)
    return match.group(0) if match else texto

def node_inspetor(state: AgentState):
    print("\n" + "="*40)
    print("🕵️‍♂️ TURNO DO INSPETOR (Llama 3.1)")
    print("="*40)
    
    # 1. Invoca o modelo
    response = llm_inspetor.invoke(state['messages'] if state['messages'] else [SystemMessage(content=PROMPT_SISTEMA_INSPETOR), HumanMessage(content="Caso: Inicie a investigação do Projeto Quimera.")])
    
    # 2. SANITIZAÇÃO: Limpa o que ele escreveu para não poluir o A2A
    conteudo_puro = limpar_mensagem(response.content)
    print(f"📤 ENVELOPE ENVIADO: {conteudo_puro[:10000]}...")
    
    is_json = "jsonrpc" in conteudo_puro.lower()
    
    return {
        "messages": [AIMessage(content=conteudo_puro)], # Salva apenas a parte limpa
        "a2a_envelope": conteudo_puro,
        "sender": "inspetor",
        "is_solved": not is_json
    }

def node_arquivista(state: AgentState):
    print("\n" + "="*40)
    print("📚 TURNO DO ARQUIVISTA (Qwen 4B)")
    print("="*40)
    
    # 1. Pega apenas o envelope limpo
    envelope = state['a2a_envelope']
    
    # 2. Extrai query
    try:
        data = json.loads(envelope)
        query = data["params"].get("query", envelope)
    except:
        query = envelope
        
    print(f"📥 PROCESSANDO REQUISIÇÃO: {query}")

    # 3. Executa MCP
    llm_mcp = llm_arquivista.bind_tools(catalogo_tool, tool_choice="required")
    res_ia = llm_mcp.invoke([SystemMessage(content=PROMPT_SISTEMA_ARQUIVISTA), HumanMessage(content=query)])
    
    resultado_mcp = "Nenhum dado novo."
    if res_ia.tool_calls:
        for call in res_ia.tool_calls:
            resultado_mcp = catalogo_britanico_mcp.invoke(call["args"])
    
    # 4. Resposta Rabugenta
    if query.lower() in str(state['messages']).lower():
        msg_final = f"JÁ DISSE! {resultado_mcp}. Não gaste minha tinta com repetições!"
    else:
        msg_final = resultado_mcp

    res_a2a = json.dumps({"jsonrpc": "2.0", "result": msg_final, "id": 1})
    print(f"📤 RESPOSTA A2A: {res_a2a}")
    
    return {
        "messages": [HumanMessage(content=f"ARQUIVISTA diz: {res_a2a}")],
        "a2a_envelope": res_a2a,
        "sender": "arquivista",
        "is_solved": False
    }

print("✅ Nodes (Inspetor e Arquivista) definidos e corrigidos.")

✅ Nodes (Inspetor e Arquivista) definidos e corrigidos.


In [12]:
# CÉLULA 6: Edges e Grafo

def router(state: AgentState):
    # Se o Inspetor decidiu que o caso está resolvido (não mandou JSON)
    if state["sender"] == "inspetor":
        if state["is_solved"]:
            print("🏁 Caso encerrado pelo Inspetor.")
            return END
        else:
            return "arquivista"
    
    # Se o arquivista respondeu, volta sempre para o Inspetor analisar
    if state["sender"] == "arquivista":
        return "inspetor"

# Reconfigurando o Grafo
workflow = StateGraph(AgentState)
workflow.add_node("inspetor", node_inspetor)
workflow.add_node("arquivista", node_arquivista)

workflow.set_entry_point("inspetor")
workflow.add_conditional_edges("inspetor", router)
workflow.add_edge("arquivista", "inspetor")

app = workflow.compile()
print("✅ Grafo compilado com sucesso.")

✅ Grafo compilado com sucesso.


In [13]:
# CÉLULA 7: Execução Final

print("🎬 RODANDO EXPERIMENTO: A Sociedade dos Detetives Literários\n")

inputs = {
    "messages": [], 
    "a2a_envelope": "", 
    "sender": "sistema"
}

# Usando stream para ver passo a passo
try:
    for output in app.stream(inputs):
        pass # Os prints dentro dos nodes já mostram o log
except Exception as e:
    print(f"❌ Erro na execução: {e}")

🎬 RODANDO EXPERIMENTO: A Sociedade dos Detetives Literários


🕵️‍♂️ TURNO DO INSPETOR (Llama 3.1)
📤 ENVELOPE ENVIADO: {"jsonrpc": "2.0", "method": "pesquisar", "params": {"query": "Projeto Quimera"}, "id": 1}...

📚 TURNO DO ARQUIVISTA (Qwen 4B)
📥 PROCESSANDO REQUISIÇÃO: Projeto Quimera
📤 RESPOSTA A2A: {"jsonrpc": "2.0", "result": "J\u00c1 DISSE! [MCP]: Registro: O autor 'V. Frankenstein' iniciou este projeto em 1816.. N\u00e3o gaste minha tinta com repeti\u00e7\u00f5es!", "id": 1}

🕵️‍♂️ TURNO DO INSPETOR (Llama 3.1)
📤 ENVELOPE ENVIADO: Parece que você está trabalhando com um sistema de busca ou uma API relacionada a um jogo ou ficção científica. O resultado da sua pesquisa é sobre o Projeto Quimera, mas parece que ele não foi bem recebido pelo "ARQUIVISTA", que é provavelmente um personagem do jogo.

O resultado indica que o autor "V. Frankenstein" iniciou o projeto em 1816 e pede para não gastar tinta com repetições, sugerindo que o Projeto Quimera foi uma ideia ou plano que não teve